# 37 - Blind calibration check: how reliable is each judge, really?

The 84.7%/13.6% agreement gap from notebook 35 isn't a clean signal, that review wasn't blind (both judges' reasoning was visible while resolving disagreements). This notebook fixes that.

**Part A** samples a fresh set of candidates neither judge has seen yet, and saves ONLY the query + company info, no judge output at all, for you to label blind.

**IMPORTANT: fill in `blind_sample.csv` yourself BEFORE running Part B.** If you run Part B first, or look at its output before finishing your labels, the blinding is broken and this whole exercise doesn't tell you anything new.

**Part B** (run only after you're done labeling) judges the same fresh sample with OpenAI and Claude, then compares both against your blind labels, that agreement rate is the real, unbiased calibration number.

## Part A -- generate the blind sample (safe to run any time, no judge calls here)

In [1]:
import json
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path("result/37_judge_blind_calibration")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_SIZE = 20
RANDOM_SEED = 7

pooled_df = pd.read_json("result/34_pooled_evaluation_set/pooled_candidates.json")
pooled_df = pooled_df[pooled_df["summary_trustworthy"]].reset_index(drop=True)

already_judged = json.load(open("result/35_llm_judge_ensemble/judge_cache.json"))
judged_keys = set(already_judged.keys())
pooled_df["key"] = pooled_df["query_id"].astype(str) + "::" + pooled_df["domain"]
fresh_df = pooled_df[~pooled_df["key"].isin(judged_keys)]

print(f"Fresh (never judged) candidates available: {len(fresh_df)}")

blind_sample = fresh_df.sample(n=SAMPLE_SIZE, random_state=RANDOM_SEED)[
    ["query_id", "query", "domain", "name", "summary"]
].reset_index(drop=True)
blind_sample["your_blind_label"] = ""

blind_sample_path = OUTPUT_DIR / "blind_sample.csv"
blind_sample.to_csv(blind_sample_path, index=False)
print(f"Saved {len(blind_sample)} fresh candidates -> {blind_sample_path}")
print("Fill in your_blind_label (0, 1, or 2) for each row, based only on the query, name, and summary shown.")
print("Do NOT run Part B, and do not visit the websites, until you've finished labeling all rows here.")

Fresh (never judged) candidates available: 7392
Saved 20 fresh candidates -> result/37_judge_blind_calibration/blind_sample.csv
Fill in your_blind_label (0, 1, or 2) for each row, based only on the query, name, and summary shown.
Do NOT run Part B, and do not visit the websites, until you've finished labeling all rows here.


## Part B -- run ONLY after `blind_sample.csv` is fully filled in

In [7]:
import json
import pandas as pd
from pathlib import Path
import os, time
import requests
from dotenv import load_dotenv

OUTPUT_DIR = Path("result/37_judge_blind_calibration")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

load_dotenv(override=True)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")

blind_sample = pd.read_csv(OUTPUT_DIR / "blind_sample.csv")
assert blind_sample["your_blind_label"].notna().all(), "Fill in every row of blind_sample.csv before running Part B."

JUDGE_PROMPT_TEMPLATE = """You are judging search result relevance for a company search engine.

Search query: "{query}"

Candidate company:
Name: {name}
Summary: {summary}

Rate how relevant this company is to the search query, using exactly one of these labels:
2 = highly relevant (a strong, direct match for the query)
1 = partially relevant (related but not a strong direct match)
0 = not relevant

Respond with ONLY a JSON object: {{"label": <0, 1, or 2>, "reason": "<one short sentence>"}}"""


def parse_judge_reply(text):
    try:
        start, end = text.index("{"), text.rindex("}") + 1
        parsed = json.loads(text[start:end])
        return int(parsed["label"]), parsed.get("reason", "")
    except (ValueError, KeyError, json.JSONDecodeError):
        return None, f"UNPARSEABLE: {text[:200]}"


def judge_openai(prompt):
    resp = requests.post(
        "https://api.openai.com/v1/chat/completions",
        headers={"Authorization": f"Bearer {OPENAI_API_KEY}", "Content-Type": "application/json"},
        json={"model": "gpt-4o-mini", "messages": [{"role": "user", "content": prompt}], "temperature": 0},
        timeout=60,
    )
    resp.raise_for_status()
    return parse_judge_reply(resp.json()["choices"][0]["message"]["content"])


def judge_claude(prompt):
    resp = requests.post(
        "https://api.anthropic.com/v1/messages",
        headers={"x-api-key": ANTHROPIC_API_KEY, "anthropic-version": "2023-06-01", "Content-Type": "application/json"},
        json={"model": "claude-haiku-4-5-20251001", "max_tokens": 200, "messages": [{"role": "user", "content": prompt}]},
        timeout=60,
    )
    resp.raise_for_status()
    return parse_judge_reply(resp.json()["content"][0]["text"])


openai_labels, claude_labels = [], []
for _, row in blind_sample.iterrows():
    prompt = JUDGE_PROMPT_TEMPLATE.format(query=row["query"], name=row["name"], summary=row["summary"])
    o_label, _ = judge_openai(prompt)
    c_label, _ = judge_claude(prompt)
    openai_labels.append(o_label)
    claude_labels.append(c_label)
    time.sleep(0.3)

blind_sample["openai_label"] = openai_labels
blind_sample["claude_label"] = claude_labels
blind_sample.to_csv(OUTPUT_DIR / "blind_sample_scored.csv", index=False)

In [8]:
openai_agree = (blind_sample["openai_label"] == blind_sample["your_blind_label"]).mean()
claude_agree = (blind_sample["claude_label"] == blind_sample["your_blind_label"]).mean()
print(f"Blind sample size: {len(blind_sample)}")
print(f"OpenAI matched your BLIND label : {openai_agree:.1%}")
print(f"Claude matched your BLIND label : {claude_agree:.1%}")

Blind sample size: 20
OpenAI matched your BLIND label : 75.0%
Claude matched your BLIND label : 80.0%
